# TP Procesamiento de Señales — Series de tiempo en charts de Spotify

**Idea:** tratar la evolución diaria de streams/posición de canciones en el Top de Spotify (Argentina / Global) como una serie temporal real, y aplicarle las herramientas de la materia (muestreo, Fourier, filtros, energía, correlación, forecasting).

Este notebook es la **base compartida**: cada integrante clona/copia este archivo, cambia la variable `CANCION` (o `GENERO`) por la serie que le toca, y corre el mismo pipeline sobre la suya.

Estado actual: **solo Parte 0 (carga de datos) y Parte 1 (análisis exploratorio)**. Las demás partes quedan como esqueleto para completar después.

## 0. Carga de datos

**Dataset usado:** [Spotify Charts (dhruvildave)](https://www.kaggle.com/datasets/dhruvildave/spotify-charts) — Kaggle.

Contiene, por día, el Top 200 / Viral 50 de Spotify en decenas de países (incluye `Argentina` y `Global`), con columnas: `title, rank, date, artist, url, region, chart, trend, streams`.

### Qué tenés que descargar (una sola vez, para todo el grupo)
1. Entrar a la página del dataset en Kaggle (necesita cuenta gratuita) y descargar el archivo `charts.csv`.
2. **Ojo**: el archivo completo es grande (varios cientos de MB / puede superar 1 GB descomprimido, son ~26 millones de filas de todos los países del mundo desde 2017). No hace falta que los 4 lo tengan completo.
3. Una persona lo descarga, corre la celda de "Filtrado inicial" de abajo (que se queda solo con Argentina + Global), y comparte el CSV chico resultante (`charts_ar_global.csv`, mucho más liviano) con el resto del equipo por Drive — ese es el archivo que todos van a usar de acá en adelante.
4. Poner el CSV chico en una carpeta `data/` al lado de este notebook.

Alternativa si no quieren crear cuenta en Kaggle: se puede usar la API de Kaggle (`kaggle datasets download -d dhruvildave/spotify-charts`) desde una notebook de Colab, pero igual requiere una cuenta y un token (`kaggle.json`).

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

pd.set_option('display.max_columns', None)

### Filtrado inicial (correr UNA vez, sobre el archivo grande original)

Si ya tenés el `charts_ar_global.csv` chico que compartió el equipo, saltate esta celda y andá directo a la carga chica de más abajo.

In [ ]:
# Solo la persona que descarga el CSV original corre esto una vez.
# Ajustar el path al archivo grande descargado de Kaggle.

RUTA_CSV_GRANDE = "data/charts.csv"  # <-- cambiar por el path real

# Como el archivo es enorme, lo leemos en chunks y nos quedamos solo
# con las filas de Argentina y Global.
chunks = pd.read_csv(RUTA_CSV_GRANDE, chunksize=500_000)

filtrado = []
for chunk in chunks:
    filtrado.append(chunk[chunk["region"].isin(["Argentina", "Global"])])

df_ar_global = pd.concat(filtrado, ignore_index=True)
df_ar_global.to_csv("data/charts_ar_global.csv", index=False)
print(df_ar_global.shape)

### Carga del CSV chico (esto lo corre todo el equipo)

In [ ]:
RUTA_CSV_CHICO = "data/charts_ar_global.csv"

df = pd.read_csv(RUTA_CSV_CHICO, parse_dates=["date"])
df = df.sort_values("date")
df.head()

In [ ]:
print("Rango de fechas:", df["date"].min(), "a", df["date"].max())
print("Regiones:", df["region"].unique())
print("Tipos de chart:", df["chart"].unique())
print("Cantidad de canciones distintas:", df["title"].nunique())
print("Cantidad de artistas distintos:", df["artist"].nunique())

## 1. Análisis exploratorio

**Cada integrante define acá su propia serie** (una canción puntual, o un género agregado) y repite esta sección con la suya.

Elegir UNA de las dos formas de armar la serie:
- **Por canción**: streams (o rank) día a día de un track puntual mientras estuvo en el chart.
- **Por género/agregado**: por ejemplo cantidad de canciones de un género en el Top 50 por semana (esto requiere cruzar con un dataset de géneros aparte, ver charla previa).

In [ ]:
# --- PARÁMETROS A CAMBIAR POR CADA INTEGRANTE ---
CANCION = "NOMBRE_DE_LA_CANCION"   # <-- reemplazar
REGION = "Argentina"                 # "Argentina" o "Global"
CHART = "top200"                     # "top200" o "viral50"
# --------------------------------------------------

serie = df[
    (df["title"] == CANCION) &
    (df["region"] == REGION) &
    (df["chart"] == CHART)
].sort_values("date").reset_index(drop=True)

print(f"Cantidad de días en el chart: {len(serie)}")
serie[["date", "rank", "streams"]].head()

### 1.1 Forma de la curva (equivalente a la "forma de onda")

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(12, 6), sharex=True)

axes[0].plot(serie["date"], serie["streams"])
axes[0].set_title(f"Streams diarios — {CANCION} ({REGION})")
axes[0].set_ylabel("Streams")

axes[1].plot(serie["date"], serie["rank"], color="orange")
axes[1].invert_yaxis()  # rank 1 = mejor puesto, va arriba
axes[1].set_title("Posición en el chart (1 = mejor)")
axes[1].set_ylabel("Rank")
axes[1].set_xlabel("Fecha")

plt.tight_layout()
plt.show()

### 1.2 Estadísticos descriptivos (media, desvío, rango, histograma)

In [ ]:
media = serie["streams"].mean()
desvio = serie["streams"].std()
rango = serie["streams"].max() - serie["streams"].min()

print(f"Media de streams:  {media:,.0f}")
print(f"Desvío estándar:   {desvio:,.0f}")
print(f"Rango:             {rango:,.0f}")

plt.figure(figsize=(8, 4))
plt.hist(serie["streams"], bins=30)
plt.title(f"Histograma de streams diarios — {CANCION}")
plt.xlabel("Streams")
plt.ylabel("Frecuencia")
plt.show()

### 1.3 Continuidad de la serie: ¿hay "silencios" (días fuera del chart)?

Si la canción salió y volvió a entrar al Top 200, van a faltar fechas intermedias — el equivalente a un "silencio" en la señal de audio.

In [ ]:
fechas_completas = pd.date_range(serie["date"].min(), serie["date"].max(), freq="D")
faltantes = fechas_completas.difference(serie["date"])

print(f"Días totales del período: {len(fechas_completas)}")
print(f"Días efectivamente en el chart: {len(serie)}")
print(f"Días 'silencio' (fuera del chart): {len(faltantes)}")

### 1.4 Cambios bruscos (saltos día a día)

Diferencia entre muestras consecutivas — picos grandes acá suelen ser un salto viral, una mención en redes, o un cambio de temporada (ej. Navidad).

In [ ]:
serie["diferencia_diaria"] = serie["streams"].diff()

plt.figure(figsize=(12, 4))
plt.plot(serie["date"], serie["diferencia_diaria"])
plt.axhline(0, color="gray", linewidth=0.8)
plt.title("Cambio de streams día a día (detección de saltos/cambios bruscos)")
plt.xlabel("Fecha")
plt.ylabel("Δ streams")
plt.show()

top_saltos = serie.reindex(serie["diferencia_diaria"].abs().sort_values(ascending=False).index).head(5)
top_saltos[["date", "streams", "diferencia_diaria"]]

---
## 2. Dominio tiempo-frecuencia (Fourier)
*(pendiente — acá va la FFT de la serie, buscar periodicidad semanal, y si eligieron una canción navideña, la componente anual)*

## 3. Filtrado
*(pendiente — medias móviles MA7 / MA21 para separar componente semanal y tendencia, filtro pasa-banda alrededor de 1/7 días)*

## 4. Energía de la señal
*(pendiente)*

## 5. Comparación / correlación entre series del equipo
*(pendiente — acá se juntan las 4 series del grupo)*

## 6. Estacionariedad y forecasting (ARIMA / Prophet)
*(pendiente)*

## 7. Conclusiones
*(pendiente)*